In [4]:
import numpy as np
from scipy import sparse

# 1. Load the files
X = sparse.load_npz('/Users/duncanpark/10-faers-foundation-model/X_train_sparse.npz')
Y = sparse.load_npz('/Users/duncanpark/10-faers-foundation-model/Y_train_sparse.npz')

# Find the first patient row that actually has a high number of active bits 
# (This guarantees we pick a patient taking multiple valid drugs)
row_sums = np.array(X.sum(axis=1)).flatten()
perfect_row_idx = int(np.where(row_sums > 100)[0][0]) 

print("=" * 60)
print("             FAERS PREPROCESSING SUCCESS SNAPSHOT          ")
print("=" * 60)
print(f"Total Patient Records Processed : {X.shape[0]:,}")
print(f"Input Feature Dimensions (X)   : {X.shape[1]:,} elements per patient")
print(f"Target Label Dimensions (Y)     : {Y.shape[1]:,} unique ADR columns")
print("-" * 60)
print(f"X Matrix Storage Data Type     : {X.dtype}")
print(f"Y Matrix Storage Data Type     : {Y.dtype}")
print("-" * 60)

# Verify the 5-slot layout using our active patient row
fp_size = 3095
dense_sample = X[perfect_row_idx].toarray().flatten()

print(f"VERIFICATION OF 5-SLOT COCKTAIL ARCHITECTURE (Patient Row {perfect_row_idx}):")
for slot in range(5):
    start = slot * fp_size
    end = start + fp_size
    slot_vector = dense_sample[start:end]
    print(f"  -> Slot {slot+1} (Indices {start:5d} to {end:5d}): {int(slot_vector.sum()):2d} active chemical bits.")
print("=" * 60)

             FAERS PREPROCESSING SUCCESS SNAPSHOT          
Total Patient Records Processed : 14,806,532
Input Feature Dimensions (X)   : 15,475 elements per patient
Target Label Dimensions (Y)     : 4,011,363 unique ADR columns
------------------------------------------------------------
X Matrix Storage Data Type     : uint8
Y Matrix Storage Data Type     : uint8
------------------------------------------------------------
VERIFICATION OF 5-SLOT COCKTAIL ARCHITECTURE (Patient Row 36):
  -> Slot 1 (Indices     0 to  3095): 119 active chemical bits.
  -> Slot 2 (Indices  3095 to  6190):  0 active chemical bits.
  -> Slot 3 (Indices  6190 to  9285):  0 active chemical bits.
  -> Slot 4 (Indices  9285 to 12380):  0 active chemical bits.
  -> Slot 5 (Indices 12380 to 15475):  0 active chemical bits.


In [2]:
from scipy import sparse
# Check how many total '1' bits exist across your entire 14.8 million rows
total_ones = X.nnz 
print(f"Total active 1s in the entire matrix: {total_ones:,}")

if total_ones == 0:
    print("ALERT: The entire matrix is empty! We have a mapping bug.")
else:
    print("Phew! The matrix has data. Row 0 was just an unmapped or padded outlier.")

Total active 1s in the entire matrix: 1,488,883
Phew! The matrix has data. Row 0 was just an unmapped or padded outlier.
